# xnb ipywidgets demo

```
xnb demo/widgets_demo.ipynb
```

xnb saves the final widget state in `metadata.widgets`. Open `widgets_demo.out.ipynb` in JupyterLab or nbviewer to see the widgets rendered with the values they had at the end of the run. It is a snapshot: no kernel is attached, so Python callbacks don't run.

In [ ]:
import ipywidgets as w
slider = w.IntSlider(value=3, min=0, max=10, description='count')
slider

Values changed by code after display are saved (the slider above ends at 8).

In [ ]:
slider.value = 8
progress = w.FloatProgress(value=0.75, min=0, max=1, description='progress')
progress

## Containers

In [ ]:
w.VBox([
    w.HTML('<b>Settings</b>'),
    w.HBox([w.Text(value='xnb', description='name'), w.Checkbox(value=True, description='enabled')]),
    w.Dropdown(options=['small', 'medium', 'large'], value='medium', description='size'),
    w.ToggleButtons(options=['a', 'b', 'c'], value='b'),
])

## Frontend links (still work in the saved notebook)

`jslink` lives in the browser, so moving one slider moves the other even without a kernel.

In [ ]:
a = w.FloatSlider(description='a')
b = w.FloatSlider(description='b')
w.jslink((a, 'value'), (b, 'value'))
w.VBox([a, b])

## Binary data (Image widget)

In [ ]:
import numpy as np, zlib, struct

def png(pixels):
    h, wd, _ = pixels.shape
    raw = b''.join(b'\x00' + pixels[y].tobytes() for y in range(h))
    chunk = lambda t, d: struct.pack('>I', len(d)) + t + d + struct.pack('>I', zlib.crc32(t + d) & 0xffffffff)
    return b'\x89PNG\r\n\x1a\n' + chunk(b'IHDR', struct.pack('>IIBBBBB', wd, h, 8, 2, 0, 0, 0)) + chunk(b'IDAT', zlib.compress(raw)) + chunk(b'IEND', b'')

y, x = np.mgrid[0:64, 0:128]
img = np.stack([x * 2, y * 4, 255 - x * 2], axis=-1).astype(np.uint8)
w.Image(value=png(img), format='png', width=256)

## Closed widgets are not saved

In [ ]:
temp = w.Button(description='temporary')
display(temp)
temp.close()
print('closed:', temp.comm is None)